In [ ]:
import sys; sys.path.append('..')
sys.path.append('../curved_linesearch/')
import MeshFEM, mesh, mesh_energy, benchmark, viewer, py_newton_optimizer

import numpy as np
import igl

import matplotlib
from matplotlib import pyplot as plt

In [ ]:
import sim_utils, param_utils
import extra_utils, opt_utils

In [ ]:
from curved_linesearch import visualization

In [ ]:
import newton_flow
import newton_flow_utils as nfu

In [ ]:
import rotation_strain_extrapolation
from Benchmark import helper_funcs

# Newton Flow Problem and its settings

In [ ]:
model = 'Hilbert2.off'

In [ ]:
m = helper_funcs.read_mesh(f'../../../Models/TableOneModels/{model}')
print(f"Model: {model} Vertices: {m.numVertices()}")
print(f"Model: {model} Elements: {m.numElements()}")

In [ ]:
uv = mesh_energy.NodalVars(m, 2)
m_2d = mesh.Mesh(np.zeros((m.numVertices(),2)), m.elements())
m_2d.reembedElements(m.vertices())


m_init_2d = mesh.Mesh('ToysMesh/Hilbert_init_2d.obj')
uv.setVars(m_init_2d.vertices().ravel())
nf = newton_flow.symmetric_dirichlet(m_2d, uv)

In [ ]:
nf.projectionSmoothingEpsilon = 0 # 1e-4 # 1e-8

In [ ]:
prob = py_newton_optimizer.NewtonMultiobjectiveProblem(uv, [nf])

In [ ]:
nf.elementHessianShift = 1e-8
prob.hessianShift = 0
prob.useRelativeHessianShift = False

In [ ]:
FIX_VARS = False
always_project = False

In [ ]:
opt = prob.optimizer()
opt.options.hessianProjectionController.startWithProjectionActive = False
opt.options.hessianProjectionController.numProjectionStepsBeforeDisable = 1
opt.options.hessianProjectionController.numConsecutiveIndefiniteStepsBeforeEnable = 0
if always_project: opt.options.hessianProjectionController = py_newton_optimizer.HessianProjectionAlways()
opt.options.niter = 200

In [ ]:
x_init = prob.getVars()

In [ ]:
prob.energy()

## Flip avoiding linesearch

In [ ]:
import flip_avoiding_step_length
prob.initialFeasibleStepLengthComputer = flip_avoiding_step_length.FlipAvoidingStepLength(m_2d.elements())
prob.initialFeasibleStepLengthComputer.backoffFactor = 0.95

In [ ]:
fasl = flip_avoiding_step_length.FlipAvoidingStepLength(m_2d.elements())

# Construct Extrapolate Method Class

In [ ]:
linear_extrapolator = extra_utils.LinearExtrapolator()

In [ ]:
pade_extrapolator = extra_utils.PadeExtrapolator(prob, opt, 14)

In [ ]:
RS_extrapolator = rotation_strain_extrapolation.RSNewtonFlowExtrapolator(m_2d)

In [ ]:
# brek

# Newton Optimize

In [ ]:
line_search_method = opt_utils.BruteForceLinesearch(alpha_step_size=0.01)

In [ ]:
# benchmark.reset()
# opt.optimize()
# benchmark.report()


In [ ]:
benchmark.reset()
vertices_list = opt_utils.newton_extrapolate(opt, linear_extrapolator, line_search_method, max_iters=10, verbose=True)
benchmark.report()

In [ ]:
exp_line_search = opt_utils.ExpLinesearch(alpha_step_size=0.1)

In [ ]:
max_alpha = 10
exp_line_search.alpha_step_size = 0.1
exp_line_search.max_alpha = max_alpha

In [ ]:
optimal_alpha_list = []
def customCallback(prob, iter_count, alpha):
    optimal_alpha_list.append(alpha)

In [ ]:
benchmark.reset()
vertices_list = opt_utils.newton_extrapolate(opt, RS_extrapolator, exp_line_search, post_step_cb=customCallback, 
                                             grad_tol=2e-8, max_iters=100, verbose=True)
benchmark.report()

In [ ]:
vertices_list.shape

In [ ]:
len(optimal_alpha_list)

In [ ]:
final_uv = vertices_list[-1]

In [ ]:
initial_uv = vertices_list[0]

In [ ]:
brek

# UV Viewer

In [ ]:
uv_final = mesh.Mesh(final_uv, m.elements())

In [ ]:
uv_viewer = viewer.Viewer(uv_final, wireframe=True)
uv_viewer.show()

## Flow Visualization

In [ ]:
fv = vertices_list

In [ ]:
extrapolation_dist = max_alpha
constant_speed = True
num_frames = min(500, len(fv))

methods = [(1, nfu.eval_trajectory_taylor, 'Newton'),
           (2, nfu.eval_trajectory_taylor, 'Deg 2 Taylor'),
           (3, nfu.eval_trajectory_taylor, 'Deg 3 Taylor'),
           (1, RS_extrapolator, 'Poisson'),
           (14, nfu.eval_trajectory_vector_pade, 'Pade 14'),
           (19, nfu.eval_trajectory_vector_pade, 'Pade 19')
]

methods = [
    (1, nfu.eval_trajectory_taylor, 'Newton'),
    (1, RS_extrapolator, 'Poisson'),
]


ff = lambda i:  visualization.flow_frame(i, opt, fv, extrapolation_dist, constant_speed,
                         extrapolation_method_list=methods, truncate=True, corners_only=True)

In [ ]:
# visualization.writeVideo('videos/Poisson_vis_traj.mp4',num_frames,ff)

## plot alpha functions

In [ ]:
def plot_optimal_alpha(axs, ind):
    plt.sca(axs[1])
    plt.axvline(x=optimal_alpha_list[ind], color='r', lw=1, ls='--')
    return plt

def plot_cached_alphas(axs, ind):
    plt.sca(axs[1])
    # line search configure
    line_search_func = exp_line_search
    extrapolator = RS_extrapolator
    
    x = fv[ind].ravel()
    prob.setVars(x)
    d = opt.newton_step()
    
    # Linesearch prepare
    extrapolator.linesearch_begin(x, d)
    def f(alpha):
        x_new = extrapolator.linesearch_eval(alpha)
        o = prob.objectiveAtVars(x_new.ravel())
        return o
    optimal_alpha = line_search_func(f, x, d)
    alphas_cached = list(line_search_func.cache.keys())
    
    # plot cached alphas
    for alpha in alphas_cached:
        plt.axvline(x=alpha, color='#B07AA1', lw=1, ls='--')
    # plot optimal alpha
    plt.axvline(x=optimal_alpha, color='r', lw=1, ls='--')
    
    # prob.setVars to original?
    return plt

In [ ]:
i = 17
axs = ff(i)
# line_search begin call first, and copy f(alpha) and then do line_search eval
# plt.sca(axs[1])
# plt.axvline(x=4, color='r', lw=1, ls='--')
plt = plot_cached_alphas(axs, i)

## Functions for augmented alphas

In [ ]:
import video_writer

def writeVideo_plot_alphas(path, num_frames, plot_frame, plot_alpha_func, skipFrame=1, framerate=30):    
    from ipywidgets import IntProgress
    from IPython.display import display
    progress = IntProgress(min=0, max=num_frames)
    display(progress)
    axs = plot_frame(0)
    plt = plot_alpha_func(axs, 0)
    vw = video_writer.PlotVideoWriter(path, plt.gcf(), dpi=150, quality='-crf 10', tight_layout=False, framerate=framerate)
    plt.close()
    for frame in range(0, num_frames, skipFrame):
        axs = plot_frame(frame)
        plt = plot_alpha_func(axs, frame)
        vw.writeFrame(plt.gcf())
        plt.close()
        progress.value = frame + 1

In [ ]:
#writeVideo_plot_alphas('videos/poisson_cached_alphas.mp4', num_frames, ff, plot_cached_alphas)